# Recopilación de datos

Este notebook carga el archivo original, separa el diagnóstico y el identificador, crea la variable objetivo y guarda el conjunto que utilizará el análisis exploratorio.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

def localizar_raiz_proyecto():
    """Localiza la carpeta principal tanto desde VS Code como desde Jupyter."""
    carpeta_actual = Path.cwd().resolve()

    if (carpeta_actual / "datos").exists():
        return carpeta_actual

    if (carpeta_actual.parent / "datos").exists():
        return carpeta_actual.parent

    raise FileNotFoundError(
        "No se encuentra la carpeta 'datos'. Abre el notebook desde el repositorio."
    )


RAIZ_PROYECTO = localizar_raiz_proyecto()

In [9]:
RUTA_ORIGINAL = RAIZ_PROYECTO / "datos" / "originales" / "thyroid0387.data"
RUTA_SALIDA = RAIZ_PROYECTO / "datos" / "intermedios" / "dataset_recopilado.csv"

RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

print("Archivo de entrada:", RUTA_ORIGINAL)
print("Archivo de salida:", RUTA_SALIDA)

Archivo de entrada: C:\Users\frana\Desktop\codigo\prediccion-enfermedades-tiroideas\datos\originales\thyroid0387.data
Archivo de salida: C:\Users\frana\Desktop\codigo\prediccion-enfermedades-tiroideas\datos\intermedios\dataset_recopilado.csv


In [10]:
columnas = [
    "edad",
    "sexo",
    "tratamiento_con_tiroxina",
    "consulta_sobre_tiroxina",
    "medicacion_antitiroidea",
    "enfermo",
    "embarazada",
    "cirugia_tiroidea",
    "tratamiento_I131",
    "consulta_hipotiroidismo",
    "consulta_hipertiroidismo",
    "litio",
    "bocio",
    "tumor",
    "hipopituitarismo",
    "trastorno_psiquiatrico",
    "TSH_medida",
    "TSH",
    "T3_medida",
    "T3",
    "TT4_medida",
    "TT4",
    "T4U_medida",
    "T4U",
    "FTI_medido",
    "FTI",
    "TBG_medida",
    "TBG",
    "fuente",
    "diagnostico_id"
]

In [11]:
# Lectura del conjunto de datos original
df = pd.read_csv(
    RUTA_ORIGINAL,
    header=None,
    names=columnas,
    dtype=str,
    na_values="?"
)

print("Dimensiones originales:", df.shape)
display(df.head())

Dimensiones originales: (9172, 30)


,edad,sexo,tratamiento_con_tiroxina,consulta_sobre_tiroxina,medicacion_antitiroidea,enfermo,embarazada,cirugia_tiroidea,tratamiento_I131,consulta_hipotiroidismo,...,TT4_medida,TT4,T4U_medida,T4U,FTI_medido,FTI,TBG_medida,TBG,fuente,diagnostico_id
0,29,F,f,f,f,f,f,f,f,t,...,f,NaN,f,NaN,f,NaN,f,NaN,other,-[840801013]
1,29,F,f,f,f,f,f,f,f,f,...,t,128,f,NaN,f,NaN,f,NaN,other,-[840801014]
2,41,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,f,NaN,t,11,other,-[840801042]
3,36,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,f,NaN,t,26,other,-[840803046]
4,32,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,f,NaN,t,36,other,S[840803047]


In [12]:
# Separar el diagnóstico y el identificador
# Ejemplo original: -[840801013]
df["diagnostico"] = df["diagnostico_id"].str.extract(r"^([^\[]+)")
df["id"] = df["diagnostico_id"].str.extract(r"\[(.*?)\]")

In [13]:
# Eliminar la columna original, ya que ya la hemos separado
df = df.drop(columns=["diagnostico_id"])

In [14]:
# Sustituir los interrogantes por valores ausentes
df = df.replace("?", np.nan)

In [15]:
# Grupos de diagnósticos
diagnosticos_hipertiroidismo = set("ABCD")
diagnosticos_hipotiroidismo = set("EFGH")


def crear_clase(diagnostico):
    if pd.isna(diagnostico):
        return "otro"

    # En diagnósticos como C|I, la documentación indica
    # que la letra situada a la derecha es la más probable.
    if "|" in diagnostico:
        diagnostico = diagnostico.split("|")[-1]

    letras_diagnostico = set(diagnostico)

    if letras_diagnostico & diagnosticos_hipertiroidismo:
        return "hipertiroidismo"

    if letras_diagnostico & diagnosticos_hipotiroidismo:
        return "hipotiroidismo"

    return "otro"

In [16]:
# Crear la variable objetivo
df["clase"] = df["diagnostico"].apply(crear_clase)

In [17]:
# Comprobación del resultado
print("Dimensiones:", df.shape)
print("Distribución de clases:")
display(df["clase"].value_counts().to_frame())
display(df.head())


Dimensiones: (9172, 32)
Distribución de clases:


,count
clase,
otro,8285
hipotiroidismo,659
hipertiroidismo,228


,edad,sexo,tratamiento_con_tiroxina,consulta_sobre_tiroxina,medicacion_antitiroidea,enfermo,embarazada,cirugia_tiroidea,tratamiento_I131,consulta_hipotiroidismo,...,T4U_medida,T4U,FTI_medido,FTI,TBG_medida,TBG,fuente,diagnostico,id,clase
0,29,F,f,f,f,f,f,f,f,t,...,f,NaN,f,NaN,f,NaN,other,-,840801013,otro
1,29,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,f,NaN,other,-,840801014,otro
2,41,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,t,11,other,-,840801042,otro
3,36,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,t,26,other,-,840803046,otro
4,32,F,f,f,f,f,f,f,f,f,...,f,NaN,f,NaN,t,36,other,S,840803047,otro


## Guardado del resultado

In [18]:
# Este archivo será la entrada del notebook de análisis exploratorio
df.to_csv(RUTA_SALIDA, index=False)

print("Conjunto recopilado guardado en:")
print(RUTA_SALIDA)

Conjunto recopilado guardado en:
C:\Users\frana\Desktop\codigo\prediccion-enfermedades-tiroideas\datos\intermedios\dataset_recopilado.csv
